# Inference LLM on notebooks

In [ ]:
!pip -q install pyngrok

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import torch

from dotenv import load_dotenv
load_dotenv("..")
import os

In [ ]:
from huggingface_hub import login

login(token="")

In [ ]:
NGROK_AUTH_TOKEN = ""

In [ ]:
model_name = "quangne/text2diagram-AceMath-1.5B-Instruct-merged-geometry3k8-8-1-1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

chat_template.jinja:   0%|          | 0.00/435 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
from pyngrok import ngrok

app = FastAPI()

class Request(BaseModel):
    prompt: str

def _prepare_inference_context():
    model.eval()
    use_cuda = torch.cuda.is_available() and str(model.device).startswith("cuda")
    compute_dtype = torch.bfloat16 if (use_cuda and torch.cuda.is_bf16_supported()) else torch.float16

    if hasattr(model, "lm_head") and hasattr(model.lm_head, "to"):
        try:
            model.lm_head = model.lm_head.to(dtype=compute_dtype)
        except Exception:
            pass

    return use_cuda, compute_dtype


def generate_dsl(
    messages: list,
    max_new_tokens: int = 256,
    use_cuda: bool = False,
    compute_dtype: torch.dtype = torch.float16,
) -> str:
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        if use_cuda:
            with torch.autocast(device_type="cuda", dtype=compute_dtype):
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    repetition_penalty=1.08,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.pad_token_id,
                    use_cache=True,
                )
        else:
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.08,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
                use_cache=True,
            )

    prompt_len = inputs["input_ids"].shape[-1]
    generated = outputs[0][prompt_len:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

@app.post("/generate")
def generate(req: Request):

    messages = [
        {"role": "user", "content": req.prompt},
    ]
    use_cuda, compute_dtype = _prepare_inference_context()
    text = generate_dsl(messages, max_new_tokens=256, use_cuda=use_cuda, compute_dtype=compute_dtype,)
    print(text)

    return {"response": text}

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000)
print("PUBLIC URL:", public_url)

server = uvicorn.Server(
    uvicorn.Config(app, host="localhost", port=8000)
)

await server.serve()

PUBLIC URL: NgrokTunnel: "https://victoria-communicable-sometimes.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [601]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:8000 (Press CTRL+C to quit)


(triangle (A B C) (isosceles A))
(define O point (circumcenter A B C))
(circle O (circumcircle A B C))
(define M point (midpoint A B))
(define N point (midpoint A C))
(define P point (midpoint B C))
(segment A B)
(segment A C)
(segment B C)
(segment M N)
(segment N C)
(segment C B)
(segment M P)
(segment N P)
(segment P B)
INFO:     ::1:47072 - "POST /generate HTTP/1.1" 200 OK
(rectangle (A B C D))
(segment A C)
(segment B D)
(define I point (inter-ll A C B D))
(define M point (midpoint A B))
(define K point (midpoint B C))
(segment M K)
(segment I M)
(segment I K)
(segment M B)
(segment M C)
(perpendicular (segment M I) (segment M K))
(parallel (segment M I) (segment K B))
(parallel (segment M K) (segment I B))
INFO:     ::1:43468 - "POST /generate HTTP/1.1" 200 OK
(rectangle (A B C D))
(segment A C)
(segment B D)
(define I point (inter-ll A C B D))
(define M point (midpoint A B))
(define K point (midpoint B C))
(segment M K)
(segment I M)
(segment I K)
(segment M B)
(perpendicular (s

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [601]
